# CS 6470 A4 — Ensembles and error slices

**60 points** · Module 9 · plan for about 3–5 focused hours

**Before you submit, rename this file to `CS6470_A4_LastName_FirstName.ipynb`.**
Open it in [Google Colab](https://colab.research.google.com/) (File → Upload notebook) or local Jupyter (Python 3.10+).

## How this notebook works

- It has **4 numbered tasks**, top to bottom. Cells marked **`# YOUR CODE`** and Markdown cells marked **YOUR ANSWER** are the graded work. Cells marked *Runs as-is* are provided — run them and read them.
- Every coding task ends with a **Checkpoint** describing what you should see, so you always know whether you are on track. Small differences (±0.01–0.02 on scores) are normal.
- **Hints name the tool** (class, argument, or doc link), not a line to paste. Newer to Python? Follow the Hints. Comfortable? Try each task from the description first. Experienced? There is an ungraded stretch at the end of the assignment page.
- A `# YOUR CODE` cell raises `NotImplementedError` until you replace that line — that is the notebook telling you where to work, not a bug.
- Before submitting: **Runtime → Restart and run all.** Every cell must run with no errors and no `NotImplementedError` left.


## Setup: the A3 protocol *(runs as-is)*

Same table (**German Credit**), same split (`random_state=42`), same preprocessing as A3 — provided, so every number you produce today is directly comparable to last module's. New idea #1: **ensembles** (many imperfect trees voting). New idea #2: **error slices** (an overall score can hide a failing subgroup).


In [1]:
# Runs as-is — A3 protocol: load, split, preprocessing.
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import f1_score

data = fetch_openml("credit-g", version=1, as_frame=True, parser="auto")
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
])
print("train:", X_train.shape, "test:", X_test.shape)


train: (800, 20) test: (200, 20)


## Task 1 of 4 — Two ensembles vs the simple model

In code:

1. Build three pipelines on the shared `pre`: `LogisticRegression(max_iter=1000)` (the fixed simple baseline, using the A3 protocol), `RandomForestClassifier(n_estimators=200, random_state=42)`, and `HistGradientBoostingClassifier(random_state=42)`.
2. Fit each on the training split; collect **test macro-F1** for all three into one small table (a `pd.Series` printed is fine).
3. Keep the fitted random forest in a variable — Task 2 slices it.

**Hints.** Tools only:
- Three pipelines on the shared `pre`: `LogisticRegression(max_iter=1000)`, `RandomForestClassifier(n_estimators=200, random_state=42)`, `HistGradientBoostingClassifier(random_state=42)`.
- Score with [`f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html) (`average="macro", labels=["bad", "good"], zero_division=0`) on the test split. A `pd.Series` of the three numbers is enough.
- Keep the fitted forest in a variable for Task 2.

**Checkpoint.** Show the requested table and interpret the observed differences. An ensemble is not required to win, and a tiny group score is not conclusive evidence of a stable disparity.


In [2]:
# YOUR CODE — Task 1. Write one small block under each comment.

# 1. Three pipelines on pre (logistic, random forest, boosting)
logreg = Pipeline([
    ("pre", pre),
    ("clf", LogisticRegression(max_iter=1000))])

randf = Pipeline([
    ("pre", pre),
    ("clf", RandomForestClassifier(n_estimators=200, random_state=42))])

gradb = Pipeline([
    ("pre", pre),
    ("clf", HistGradientBoostingClassifier(random_state=42))])

# 2. Fit each on train; collect test macro-F1 in one small table
logreg.fit(X_train, y_train)
randf.fit(X_train, y_train)
gradb.fit(X_train, y_train)

pd.Series({
    "Logistic Regression F1": f1_score(y_test, logreg.predict(X_test), average="macro", labels=["bad", "good"], zero_division=0),
    "Random Forest F1": f1_score(y_test, randf.predict(X_test), average="macro", labels=["bad", "good"], zero_division=0),
    "Gradient Boosting F1": f1_score(y_test, gradb.predict(X_test), average="macro", labels=["bad", "good"], zero_division=0)
})

# 3. Keep the fitted random forest (Task 2 slices it)


Logistic Regression F1    0.636151
Random Forest F1          0.675156
Gradient Boosting F1      0.657097
dtype: float64

## Task 2 of 4 — Score one error slice

The overall number is an average — some groups live above it, some below. Cut the test set by one column and score each group.

In code, for the random forest:

1. Make predictions **aligned to the test index**: `pred = pd.Series(rf.predict(X_test), index=X_test.index)` — this is the trick that makes slice lookups safe.
2. Group the test rows by `checking_status` (or another column you prefer).
3. For every group with **at least 20 test rows**, print: group name, group size, group macro-F1.
4. Print the overall macro-F1 next to them for comparison.

**Hints.** Tools only:
- Align predictions: `pd.Series(..., index=X_test.index)` so later `.loc` lookups are safe.
- [`groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) on `checking_status` (`observed=True`); skip groups with fewer than 20 test rows.
- Score each remaining group with `f1_score` (`average="macro", labels=["bad", "good"], zero_division=0`) on the aligned slices; print overall next to them.

**Checkpoint.** Show the requested table and interpret the observed differences. An ensemble is not required to win, and a tiny group score is not conclusive evidence of a stable disparity.


In [ ]:
# YOUR CODE — Task 2. Write one small block under each comment.

# 1. Predictions aligned to the test index
pred = pd.Series(randf.predict(X_test), index=X_test.index)

# 2. Group test rows by checking_status (or another column)

grouped = X_test.groupby("checking_status", observed=True)
groups = grouped.size().to_frame("size")
groups = groups[groups["size"] >= 20]

# 3. For each group with n >= 20: print name, size, group macro-F1 

for name, group in grouped:
    if len(group) >= 20:
        group_y_true = y_test.loc[group.index]
        group_y_pred = pred.loc[group.index]
        group_f1 = f1_score(group_y_true, group_y_pred, average="macro", labels=["bad", "good"], zero_division=0)
        print(f"Group: {name}, Size: {len(group)}, Macro-F1: {group_f1:.4f}")

# 4. Print overall macro-F1 beside them
print(f"Overall Macro-F1: {f1_score(y_test, pred, average='macro', labels=['bad', 'good'], zero_division=0):.4f}")

Group: 0<=X<200, Size: 68, Macro-F1: 0.6769
Group: <0, Size: 49, Macro-F1: 0.6325
Group: no checking, Size: 71, Macro-F1: 0.4662
Overall Macro-F1: 0.6752


## Task 3 of 4 — Read the slice — YOUR ANSWER

**Q1.** Which group is weakest, and is it a small corner case or a big share of the test set?

**TODO** The no-checking group is the weakest, with a macro-F1 of 0.4662, and contains 71 of the 200 test cases. That is a little under 36% of the test set, so it represents a large share of the test set.

**Q2.** Would a real lender care about this particular group being scored badly? Why or why not?

**TODO** A real lender would care about this group because the no-checking category means that the applicant is recorded as not having a checking account. The model may make less reliable credit assessments for these applicants, potentially leading to qualified applicants being denied credit or applicants who may not repay being approved. However, a low macro-F1 alone doesn't tell us whether the model makes one type of mistake more often than another. I would need to look at the specific prediction errors to understand that.

**Q3.** What single number goes on a monitoring dashboard because of this table, and how often should someone look at it?

**TODO:** I would use the no-checking group's macro-F1 of 0.4662 on the monitoring dashboard and review it weekly to see whether the model's performance for this group gets better or worse.


## Task 4 of 4 — Memo (≤1 page) — YOUR ANSWER

For a manager. Replace each **TODO**.

### Bias vs variance, in words

**TODO:** A single decision tree can learn very detailed patterns from the training data, but it may also learn noise and become less reliable on new data. This is a high-variance model. A random forest combines many trees trained on different samples and feature subsets, allowing their predictions to average out. This usually produces more stable predictions and lowers variance while maintaining useful predictive power.

### What the slice table changes

**TODO:** The overall score of 0.6752 out of 1 gives us a general idea of how well the model is doing, but it doesn't tell us how well it works for every group of applicants. The slice table showed that the model did much worse for applicants with no checking account, with a score of 0.4662 out of 1. I would recommend looking more closely at the mistakes it makes with this group to understand what is going wrong and how we could improve the model's performance for this group. Until we understand the problem, I would keep a person involved in reviewing decisions for this group instead of relying on the model alone.

### The monitor I would install

**TODO:** I would monitor the no-checking group's macro-F1 of 0.4662 alongside the overall macro-F1 of 0.6752. Someone should review both scores weekly to see whether the performance gap gets better or worse. The model owner or credit-risk manager should review these metrics weekly and before any major model update. If the no-checking group's performance continues to lag behind the overall score, additional investigation or policy review should occur.

### AI disclosure

**TODO:** I used ChatGPT as a tutor to help me understand the topics, troubleshoot my code, and check my spelling and grammar. All the writing and code are mine.


## Before you submit

- [ ] File renamed to `CS6470_A4_LastName_FirstName.ipynb`
- [ ] **Runtime → Restart and run all** — every cell ran, no errors, no `NotImplementedError` anywhere
- [ ] All 4 tasks done: every `# YOUR CODE` cell has your code, every **YOUR ANSWER** / **TODO** prompt has your words
- [ ] Memo written in the Markdown cells above (it stays inside this notebook — no separate PDF)
- [ ] AI-use disclosure filled in

Upload the `.ipynb` file to this assignment on Canvas (in Colab: File → Download → Download .ipynb).

Integrity: the notebook and memo must be yours. AI may help explain an error; it may not be the submitted analysis.
